# DSCI 552 Homework 3

**Name:** Yufan Yao  
**USC ID:** 1450125536  
**Github Name:** yufanyao-cloud



In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# Change this path only if your AReM folder is stored somewhere else.
DATA_DIR = Path("../data/AReM")

ACTIVITIES = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking",
]

COLUMNS = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]

SIGNAL_COLUMNS = COLUMNS[1:]

print("Current working directory:", Path.cwd())
print("Data directory:", DATA_DIR.resolve())
print("Data directory exists:", DATA_DIR.exists())




Current working directory: c:\Users\yaofa\OneDrive\Desktop\Homework3_Yufan_Yao\notebook
Data directory: C:\Users\yaofa\OneDrive\Desktop\Homework3_Yufan_Yao\data\AReM
Data directory exists: True


## 1. Time Series Classification Part 1

### 1(a) Import the datasets




In [7]:
def dataset_number(file_path):
    """Extract the number from a file name such as dataset12.csv."""
    return int(file_path.stem.replace("dataset", ""))


def read_arem_file(file_path):
    """Read and clean one AReM data file."""

    df = pd.read_csv(
        file_path,
        comment="#",
        header=None,
        names=range(8),
        sep=r"[,\s]+",
        engine="python",
        skip_blank_lines=True,
    )

    # Remove the completely empty eighth column caused by a trailing comma
    df = df.dropna(axis=1, how="all")

    # A valid AReM file should contain seven columns
    if df.shape[1] != 7:
        raise ValueError(
            f"{file_path} should have 7 columns, "
            f"but {df.shape[1]} columns were found."
        )

    df.columns = COLUMNS

    # Convert all values to numbers
    df = df.apply(pd.to_numeric, errors="coerce")

    # Remove completely empty rows
    df = df.dropna(how="all").reset_index(drop=True)

    return df

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_DIR}. Put the seven activity folders inside it."
    )

file_counts = {}

for activity in ACTIVITIES:
    files = sorted(
        (DATA_DIR / activity).glob("dataset*.csv"),
        key=dataset_number,
    )
    file_counts[activity] = len(files)

file_count_table = pd.Series(
    file_counts,
    name="number_of_files",
).to_frame()

display(file_count_table)
print("Total number of instances:", file_count_table["number_of_files"].sum())




,number_of_files
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


Total number of instances: 88


The expected file counts are 7 for bending1, 6 for bending2, and 15 for
each of the other five activities. Therefore, there should be 88 instances.




In [8]:
# Check the size and missing values of every file.
quality_rows = []

for activity in ACTIVITIES:
    files = sorted(
        (DATA_DIR / activity).glob("dataset*.csv"),
        key=dataset_number,
    )

    for file_path in files:
        df = read_arem_file(file_path)
        quality_rows.append(
            {
                "activity": activity,
                "dataset": file_path.stem,
                "rows": len(df),
                "columns": df.shape[1],
                "missing_values": int(df.isna().sum().sum()),
            }
        )

quality_report = pd.DataFrame(quality_rows)

display(
    quality_report["rows"]
    .value_counts()
    .sort_index()
    .rename_axis("rows")
    .to_frame("number_of_files")
)

short_file = quality_report[quality_report["rows"] != 480]
display(short_file)

assert len(quality_report) == 88, "The dataset should contain 88 files."
assert (quality_report["rows"] == 480).sum() == 87
assert (quality_report["rows"] == 479).sum() == 1
assert (quality_report["columns"] == 7).all()
assert quality_report["missing_values"].sum() == 0

print("Data check passed: 87 files have 480 rows and one file has 479 rows.")




,number_of_files
rows,
479,1
480,87


,activity,dataset,rows,columns,missing_values
50,sitting,dataset8,479,7,0


Data check passed: 87 files have 480 rows and one file has 479 rows.


In [9]:
# Inspect one example file.
sample_file = DATA_DIR / "bending1" / "dataset1.csv"
sample_df = read_arem_file(sample_file)

print("Sample shape:", sample_df.shape)
display(sample_df.head())




Sample shape: (480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### 1(b) Training and test sets

The required split is:

- bending1 and bending2: datasets 1 and 2 are test data.
- Other activities: datasets 1, 2, and 3 are test data.
- All remaining datasets are training data.




In [10]:
def assign_set(activity, number):
    """Assign one file to the required training or test set."""
    if activity in ["bending1", "bending2"]:
        return "test" if number in [1, 2] else "train"
    return "test" if number in [1, 2, 3] else "train"


split_rows = []

for activity in ACTIVITIES:
    files = sorted(
        (DATA_DIR / activity).glob("dataset*.csv"),
        key=dataset_number,
    )

    for file_path in files:
        number = dataset_number(file_path)
        split_rows.append(
            {
                "activity": activity,
                "dataset": file_path.stem,
                "dataset_number": number,
                "set": assign_set(activity, number),
                "file_path": str(file_path),
            }
        )

split_table = pd.DataFrame(split_rows)

display(pd.crosstab(split_table["activity"], split_table["set"], margins=True))
print("Training instances:", (split_table["set"] == "train").sum())
print("Test instances:", (split_table["set"] == "test").sum())




set,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


Training instances: 69
Test instances: 19


### 1(c)(i) Time-domain features

Common time-domain features used in time-series classification include:

- minimum   
- maximum  
- mean 
- median  
- variance   
- standard deviation  
- first quartile and third quartile
- range and interquartile range
- root mean square
- skewness and kurtosis
- zero-crossing rate
- signal energy

These features summarize the level, variation, range, and shape of a signal.
They reduce a long time series to a fixed-length feature vector that can be
used by standard machine-learning algorithms.


### 1(c)(ii) Extract the required time-domain features

For each of the six time series, this question requires minimum, maximum,
mean, median, standard deviation, first quartile, and third quartile.
Therefore, each instance has $6 \times 7 = 42$ features.




In [11]:
def extract_features(df):
    """Extract the 42 required features from one instance."""
    feature_row = {}

    for series_number, column in enumerate(SIGNAL_COLUMNS, start=1):
        series = df[column].dropna()

        feature_row[f"min{series_number}"] = series.min()
        feature_row[f"max{series_number}"] = series.max()
        feature_row[f"mean{series_number}"] = series.mean()
        feature_row[f"median{series_number}"] = series.median()
        feature_row[f"std{series_number}"] = series.std(ddof=1)
        feature_row[f"1st_quart{series_number}"] = series.quantile(0.25)
        feature_row[f"3rd_quart{series_number}"] = series.quantile(0.75)

    return feature_row


feature_rows = []

for _, file_info in split_table.iterrows():
    df = read_arem_file(Path(file_info["file_path"]))
    features = extract_features(df)

    features["activity"] = file_info["activity"]
    features["dataset"] = file_info["dataset"]
    features["set"] = file_info["set"]
    feature_rows.append(features)

feature_df = pd.DataFrame(feature_rows)

metadata_columns = ["activity", "dataset", "set"]
feature_columns = [
    column for column in feature_df.columns if column not in metadata_columns
]
feature_df = feature_df[metadata_columns + feature_columns]

print("Feature dataset shape:", feature_df.shape)
print("Number of numerical features:", len(feature_columns))
display(feature_df.head())




Feature dataset shape: (88, 45)
Number of numerical features: 42


,activity,dataset,set,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,dataset1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,0.43,0.322605,0.00,0.50,4.0,29.50,19.040937,19.250,4.462952,16.00,23.2500,0.0,7.23,0.832542,0.50,0.965659,0.00,1.120,27.25,38.25,34.311292,35.0,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1,dataset2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,0.47,0.289158,0.00,0.50,2.0,29.50,20.096979,21.000,3.893737,19.00,22.2500,0.0,5.76,0.876438,0.50,1.047259,0.00,1.145,27.67,38.50,33.024583,33.0,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1,dataset3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,0.47,0.338690,0.00,0.50,6.5,29.75,22.122354,23.000,3.030943,19.75,24.0000,0.0,4.44,0.497312,0.43,0.550657,0.00,0.830,29.00,38.50,35.588458,36.0,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1,dataset4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,0.50,0.630860,0.00,1.12,8.5,30.00,22.183625,23.000,3.810469,20.50,24.3725,0.0,5.15,0.989917,0.83,0.953730,0.43,1.300,20.00,38.67,33.493917,35.0,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1,dataset5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,0.50,0.405469,0.43,0.71,3.0,28.25,19.006563,19.125,4.087107,16.50,22.0625,0.0,6.42,0.841875,0.50,0.928801,0.43,1.120,23.67,37.50,29.857083,30.0,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


The result has 88 rows and 42 numerical features. The other three columns
record the activity, dataset name, and training/test membership.




### 1(c)(iii) Standard deviations and 90% bootstrap confidence intervals




In [12]:
def bootstrap_std_ci(data, n_bootstrap=1000, confidence_level=0.90, seed=552):
    """Calculate a standard deviation and its percentile bootstrap interval."""
    values = np.asarray(data, dtype=float)
    values = values[~np.isnan(values)]

    rng = np.random.default_rng(seed)
    bootstrap_std = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_std[i] = np.std(sample, ddof=1)

    alpha = 1 - confidence_level
    lower = np.quantile(bootstrap_std, alpha / 2)
    upper = np.quantile(bootstrap_std, 1 - alpha / 2)

    return np.std(values, ddof=1), lower, upper


bootstrap_rows = []

for feature in feature_columns:
    standard_deviation, lower_bound, upper_bound = bootstrap_std_ci(
        feature_df[feature]
    )

    bootstrap_rows.append(
        {
            "feature": feature,
            "standard_deviation": standard_deviation,
            "ci_90_lower": lower_bound,
            "ci_90_upper": upper_bound,
        }
    )

bootstrap_results = pd.DataFrame(bootstrap_rows).set_index("feature")
display(bootstrap_results.round(4))




,standard_deviation,ci_90_lower,ci_90_upper
feature,,,
min1,9.5700,8.2353,10.6892
max1,4.3944,3.3326,5.3047
mean1,5.3357,4.7048,5.8863
median1,5.4401,4.7922,6.0093
std1,1.7722,1.5720,1.9438
1st_quart1,6.1536,5.5678,6.6341
3rd_quart1,5.1389,4.3299,5.8612
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6400,5.3919


### 1(c)(iv) Select three important time-domain features
I select **minimum, mean, and maximum** as the three important feature types.
The minimum and maximum describe the observed range of a signal, while the
mean represents its overall level. These features are easy to interpret and
can capture posture-related changes in RSS values.




## 2. ISLR 3.7.4

### (a)

The cubic regression is expected to have a lower training RSS because it is
more flexible. It can also become a linear model when the coefficients of
$X^2$ and $X^3$ are zero, so its training RSS cannot be higher.

### (b)

The linear regression is expected to have a lower test RSS because the true
relationship is linear. The cubic regression may fit noise in the training
data and therefore have a higher test RSS.

### (c)

The cubic regression is expected to have a lower training RSS because it is
more flexible. This is true even when the real relationship is nonlinear.

### (d)

There is not enough information to decide which model has a lower test RSS.
The answer depends on how nonlinear the real relationship is and how much
the cubic model overfits the training data.
